# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and perform basic processing on the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Keep as object, do not subscript

print("Name:", metadata.name)
print("Description:", metadata.description)
print("Published:", metadata.datePublished)
print("License:", metadata.license)
print("Spatial Coverage:", metadata.spatialCoverage)
print("Temporal Coverage:", getattr(metadata, 'temporalCoverage', 'N/A'))

## 2. Data Overview
Review available record sets, their fields, and the `@id` for each entity.

Listing the record sets and fields by `@id` helps you know what data is available for each table-like structure in the dataset.

In [ ]:
# List all record sets by @id
record_sets = [r for r in dataset.record_sets]
print(f"Number of record sets: {len(record_sets)}")
for idx, record_set in enumerate(record_sets):
    print(f"[{idx}] Record Set: {record_set.id}")
    print(f"    Name: {record_set.name}")
    print(f"    Fields:")
    for field in record_set.fields:
        print(f"     - {field.id} (name: {field.name})")
    print()

## 3. Data Extraction

Extract one or more record sets into DataFrames for easier exploration. For each record set, we reference it by its `@id`.

In [ ]:
# Collect all record set @ids for extraction
record_set_ids = [r.id for r in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Load records
    records = list(dataset.records(record_set=record_set_id))
    # Convert to DataFrame
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

for record_set_id, df in dataframes.items():
    print(f"Record set: {record_set_id}")
    print(f"Columns: {list(df.columns)}")
    print(df.head(2))
    print('-'*60)

## 4. Exploratory Data Analysis (EDA)

Now let's select a record set and numeric field for analysis.
- We'll choose the first non-empty record set that contains at least one numeric field for demonstration.

We'll filter by the numeric field, normalize it, and group by another field if available.

In [ ]:
# Find first non-empty record set with at least one numeric field
import numpy as np

numeric_types = ['Integer', 'Float', 'Number']
selected_record_set = None
selected_numeric_field_id = None
group_field_id = None
for record_set in dataset.record_sets:
    df = dataframes.get(record_set.id)
    if df is not None and not df.empty:
        # Find a numeric field by its @id
        numeric_fields = [(f.id, f.data_type) for f in record_set.fields if getattr(f, 'data_type', None) in numeric_types]
        if numeric_fields:
            selected_record_set = record_set
            selected_numeric_field_id = numeric_fields[0][0]
            # Optionally select another non-numeric field to group by
            non_numeric_fields = [f.id for f in record_set.fields if getattr(f, 'data_type', None) not in numeric_types]
            if non_numeric_fields:
                group_field_id = non_numeric_fields[0]
            break

if selected_record_set is not None:
    print(f"Selected record set: {selected_record_set.id}")
    print(f"Selected numeric field @id: {selected_numeric_field_id}")
    if group_field_id:
        print(f"Group field @id: {group_field_id}")
    df = dataframes[selected_record_set.id].copy()

    # Drop NA for numeric field, ensure correct data type
    df[selected_numeric_field_id] = pd.to_numeric(df[selected_numeric_field_id], errors='coerce')
    threshold = np.nanmean(df[selected_numeric_field_id])  # Use the mean as example threshold
    filtered_df = df[df[selected_numeric_field_id] > threshold]
    print(f"Filtered records with {selected_numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the numeric field in filtered records
    colnorm = f"{selected_numeric_field_id}_normalized"
    filtered_df[colnorm] = (filtered_df[selected_numeric_field_id] - filtered_df[selected_numeric_field_id].mean()) / filtered_df[selected_numeric_field_id].std()
    print(f"\nNormalized {selected_numeric_field_id} for filtered records:")
    print(filtered_df[[selected_numeric_field_id, colnorm]].head())

    # Group by the group_field (if present and not unique per row)
    if group_field_id and group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(group_field_id)[selected_numeric_field_id].mean()
        print(f"\nGrouped mean {selected_numeric_field_id} by {group_field_id}:")
        print(grouped.head())
else:
    print("No suitable record set with numeric fields was found in the dataset.")

## 5. Visualization

Visualize the distribution of the selected numeric field. We'll plot a histogram and, if a group field is available, compare means by groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set is not None and selected_numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[selected_numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {selected_numeric_field_id}")
    plt.xlabel(selected_numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field_id, y=selected_numeric_field_id)
        plt.title(f"{selected_numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to access a FAIR-compliant dataset described by a Croissant schema using `mlcroissant`. We programmatically discovered record sets, extracted data using `@id` references, performed basic filtering and normalization, and visualized the distribution of numeric fields. For deeper analysis, consider exploring individual field definitions, handling categorical variables, or conducting formal statistical modeling. This approach facilitates reproducible, FAIR-aligned data science workflows for robust, transparent analytics on complex research datasets.